In [1]:
# https://docs.pytorch.org/tutorials/beginner/ddp_series_intro.html

# Learnings

- In case of making code DDP, you need to:
  - `init_process_group` with rank, world_size and nccl
  - wrap the model with `DDP()`
  - when saving a checkpoint, use `model.module` instead of `model`
  - Only save the model in the first GPU as other GPUs are replicas
  - Add a `DistributedSampler` in dataloder and turn off shuffle
- `BatchNorm` needs to be replaced with `SyncBatchNorm` in DDP
- `torchrun` is better than `mp.spawn` because it allows fault-tolerant and elastic training, meaning compute resources can join and leave dynamically over the course of the job. It will gracefully restarting training from the last saved training snapshot.
  - It also takes care of the env vars, you dont have to do it manually
- In multi-node training, `RANK` refers to `NODE_RANK`. Do not use RANK for critical logic in your training job. When torchrun restarts processes after a failure or membership changes, there is no guarantee that the processes will hold the same LOCAL_RANK and RANKS.?

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyTrainDataset(Dataset):
    def __init__(self, size):
        self.size = size
        self.data = [(torch.rand(20), torch.rand(1)) for _ in range(size)]

    def __len__(self):
        return self.size

    def __getitem__(self, index):
        return self.data[index]

In [7]:
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os

In [6]:
def ddp_setup(rank, world_size):
  os.environ["MASTER_ADDR"] = "localhost"
  os.environ["MASTER_PORT"] = "8080"
  init_process_group(backend="nccl", rank=rank, world_size=world_size)

In [ ]:
class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_data: DataLoader,
        optimizer: torch.optim.Optimizer,
        gpu_id: int,
        save_every: int,
    ) -> None:
        self.gpu_id = gpu_id
        self.model = model.to(gpu_id)
        self.train_data = train_data
        self.optimizer = optimizer
        self.save_every = save_every
        self.model = DDP(model, device_ids=[gpu_id])  # DDP Change 1

    def _run_batch(self, source, targets):
        self.optimizer.zero_grad()
        output = self.model(source)
        loss = F.cross_entropy(output, targets)
        loss.backward()
        self.optimizer.step()

    def _run_epoch(self, epoch):
        b_sz = len(next(iter(self.train_data))[0])
        print(f"[GPU{self.gpu_id}] Epoch {epoch} | Batchsize: {b_sz} | Steps: {len(self.train_data)}")
        self.train_data.sampler.set_epoch(epoch)
        for source, targets in self.train_data:
            source = source.to(self.gpu_id)
            targets = targets.to(self.gpu_id)
            self._run_batch(source, targets)

    def _save_checkpoint(self, epoch):
        ckp = self.model.module.state_dict()  # DDP Change 2
        PATH = "checkpoint.pt"
        torch.save(ckp, PATH)
        print(f"Epoch {epoch} | Training checkpoint saved at {PATH}")

    def train(self, max_epochs: int):
        for epoch in range(max_epochs):
            self._run_epoch(epoch)
            if self.gpu_id == 0 and epoch % self.save_every == 0:  # DDP Change 3 (only save on rank 0 because in DDP others is redundant)
                self._save_checkpoint(epoch)

In [8]:
# Add DistributedSampler to dataloader and turn shuffle to False

In [ ]:
# Use mp.spawn to start a DP job with nproces instead of torchrun